# Ticari Orta Faz 1 - Execution Simulation

End-to-end exercise of the production pipeline against Oracle (no synthetic data, no notebook-local logic). Each section is self-contained: run the cells top to bottom or re-run a single block whenever needed.

**Sections at a glance**

- 1 / 1A : native table health and raw field distributions
- 2 / 2A : materialization into the derived feature table (42 columns)
- 3 / 3A / 3B : development, window split review, robustness comparisons
- 4 / 4A : supervised evaluation and weight tuning metrics
- 5 / 5A : promote + single / date-range scoring and label lift
- 6 / 6A : per-run monitoring bundle (score summary, band share)
- 7A : cross-run trend from the Oracle monitor history table
- 7 : operational notes

**How to read the outputs overall**

- Tables print the most important health metrics; wider payloads stay in `runtime/runs/<run_id>/`
- Plots focus on drift / share / shape changes across snapshots; colors match alert bands where relevant
- Oracle objects referenced here: `EWS_TO_FAZ1_NATIVE`, `EWS_TO_FAZ1_INPUT`, `EWS_TO_FAZ1_RESULTS`, `EWS_TO_FAZ1_DETAILS`, `EWS_TO_FAZ1_FEATURE_EFFECTS`, `EWS_TO_FAZ1_MONITOR_HISTORY`, `EWS_TO_FAZ1_OUTCOMES`

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import ks_2samp
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "engine").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from engine.config_loader import load_config, load_secrets
from engine.lifecycle import LifecycleManager
from engine.materialization import NativeMaterializer
from engine.oracle_io import OracleConnector

plt.style.use("default")

config = load_config()
secrets = load_secrets()
manager = LifecycleManager()
materializer = NativeMaterializer(config, secrets)


In [ ]:
def compact_dict(payload):
    if not isinstance(payload, dict):
        return payload
    return {key: value for key, value in payload.items() if key != "frame"}

def safe_run(label, fn, *args, **kwargs):
    try:
        result = fn(*args, **kwargs)
        print(f"{label}: OK")
        return result
    except Exception as exc:
        print(f"{label}: FAILED -> {exc}")
        return {"error": str(exc)}

def load_json_payload(path_value):
    if not path_value:
        return None
    path = Path(path_value)
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

def compute_top_percent_metrics(frame, score_column, label_column, top_percent):
    ordered = frame[[score_column, label_column]].dropna().sort_values(score_column, ascending=False).reset_index(drop=True)
    if ordered.empty:
        return {}
    top_n = max(1, int(np.ceil(len(ordered) * top_percent)))
    ordered["pred_top"] = 0
    ordered.loc[: top_n - 1, "pred_top"] = 1
    y_true = ordered[label_column].astype(int)
    y_pred = ordered["pred_top"].astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "rows": int(len(ordered)),
        "top_n": int(top_n),
        "positive_rows": int(y_true.sum()),
        "baseline_rate": float(y_true.mean()),
        "precision_at_top_percent": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall_at_top_percent": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_at_top_percent": float(f1_score(y_true, y_pred, zero_division=0)),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }

def flatten_outcome_metrics(metrics_payload):
    rows = []
    if not isinstance(metrics_payload, dict):
        return pd.DataFrame()
    primary = metrics_payload.get("primary")
    if isinstance(primary, dict):
        row = {"metric_group": "primary", **primary}
        rows.append(row)
    for key, payload in (metrics_payload.get("monitoring") or {}).items():
        if isinstance(payload, dict):
            rows.append({"metric_group": key, **payload})
    return pd.DataFrame(rows)

def draw_metric_bar(frame, *, category_column, value_columns, title):
    if frame.empty:
        print(f"{title}: veri yok")
        return
    ax = frame.set_index(category_column)[value_columns].plot(kind="bar", figsize=(10, 4))
    ax.set_title(title)
    ax.set_ylabel("value")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

def resolve_runtime_context(config, secrets):
    segment_value = config["development"]["segment_value"]
    snapshot_cfg = config["live_scoring"]["snapshot"]
    with OracleConnector(config, secrets) as ora:
        native_table = ora._qualified_table_name("native_features")
        snapshot_rows = ora._read_query(
            f"""
            SELECT DISTINCT TRUNC(SNAPSHOT_DATE) AS snapshot_date
            FROM {native_table}
            WHERE SEGMENT = :segment_value
            ORDER BY TRUNC(SNAPSHOT_DATE)
            """,
            {"segment_value": segment_value},
        )
    if snapshot_rows.empty:
        raise ValueError(
            "Native tabloda secilen segment icin snapshot bulunamadi. Once native tabloyu beslemelisin."
        )
    snapshot_values = pd.to_datetime(snapshot_rows["snapshot_date"]).sort_values().tolist()
    single_snapshot = snapshot_cfg.get("explicit_date") or pd.Timestamp(snapshot_values[-1]).date().isoformat()
    if snapshot_cfg.get("start_date") and snapshot_cfg.get("end_date"):
        range_start = snapshot_cfg["start_date"]
        range_end = snapshot_cfg["end_date"]
    else:
        tail_values = snapshot_values[-4:] if len(snapshot_values) >= 4 else snapshot_values
        range_start = pd.Timestamp(tail_values[0]).date().isoformat()
        range_end = pd.Timestamp(tail_values[-1]).date().isoformat()
    return {
        "pipeline_name": config["pipeline"]["name"],
        "segment": segment_value,
        "native_source": config["sources"]["native_features"]["oracle"]["table"],
        "derived_source": config["sources"]["input_features"]["oracle"]["table"],
        "outcomes_source": config["sources"]["outcomes"]["oracle"]["table"],
        "score_live_selector": snapshot_cfg["selector"],
        "single_snapshot": single_snapshot,
        "range_start": range_start,
        "range_end": range_end,
        "materialization_enabled": config["materialization"]["enabled"],
        "warmup_snapshots": config["materialization"]["trim_warmup_snapshots"],
    }

runtime_context = resolve_runtime_context(config, secrets)
SEGMENT = runtime_context["segment"]
SINGLE_SNAPSHOT = runtime_context["single_snapshot"]
RANGE_START = runtime_context["range_start"]
RANGE_END = runtime_context["range_end"]
TARGET_COLUMN = config["weight_optimization"]["target_column"]
TOP_PERCENT = float(config["weight_optimization"]["top_percent"])

overview = {
    **runtime_context,
}
display(pd.DataFrame([overview]))


## 1. Native Data Checks

**What it does**
Reads `EWS_TO_FAZ1_NATIVE` for the configured Faz 1 segment, prints an overview of snapshot coverage, customer count and row count, and shows 5 sample rows.

**How to read the output**

- `min_snapshot_date` / `max_snapshot_date` should match the business cadence (monthly, month-end). A gap means the upstream ETL missed a month.
- `row_count` should be roughly `customer_count * snapshots`; a large drop means customers dropped out of the cohort.
- `customer_count` reflects unique `CUSTOMER_ID` values. If the number shrinks over time compared with earlier runs, the segment filter may have tightened.
- The sample rows should contain all 23 native columns with realistic values; NULLs in critical columns (`fs_net_sales_cumulative`, `pos_monthly_volume`) mean the ETL is incomplete.

In [ ]:
with OracleConnector(config, secrets) as ora:
    native_table = ora._qualified_table_name("native_features")
    native_overview = ora._read_query(
        f"""
        SELECT
            MIN(SNAPSHOT_DATE) AS min_snapshot_date,
            MAX(SNAPSHOT_DATE) AS max_snapshot_date,
            COUNT(*) AS row_count,
            COUNT(DISTINCT CUSTOMER_ID) AS customer_count
        FROM {native_table}
        WHERE SEGMENT = :segment_value
        """,
        {"segment_value": SEGMENT},
    )
    native_sample = ora._read_query(
        f"""
        SELECT *
        FROM {native_table}
        WHERE SEGMENT = :segment_value
          AND ROWNUM <= 5
        ORDER BY SNAPSHOT_DATE DESC
        """,
        {"segment_value": SEGMENT},
    )

display(native_overview)
display(native_sample)


## 1A. Native Quality and Distribution Checks

**What it does**
Pulls a projection of the key numeric native columns and produces three artefacts:

1. A `native_missing` table ranking columns by missing share.
2. A `snapshot_density` table (rows + unique customers per snapshot).
3. Distribution plots: rows per snapshot + missing share bars + per-feature histograms.

**How to read the output**

- **Missing share** > 5 percent for a required column (for example `fs_net_sales_cumulative`) is a red flag and typically blocks downstream training.
- **Snapshot density** should be stable month over month. A sharp drop can indicate upstream data loss; a spike can indicate a backfill.
- **Rows-by-snapshot plot** should be a near-flat line; jumps larger than 10 percent deserve investigation.
- **Histograms** show the shape of each native metric. Bimodal or extreme-tail shapes that appear suddenly are early warning signs that should be correlated with the materialization layer in section 2A.

In [ ]:
with OracleConnector(config, secrets) as ora:
    native_table = ora._qualified_table_name("native_features")
    native_profile = ora._read_query(
        f"""
        SELECT SNAPSHOT_DATE, CUSTOMER_ID,
               MEMZUC_TOTAL_CASH_RISK_0_24M,
               FS_NET_SALES_CUMULATIVE,
               FS_EBITDA_CUMULATIVE,
               POS_MONTHLY_VOLUME,
               IFRS9_BEHAVIORAL_PD,
               KKB_COMMERCIAL_SCORE
        FROM {native_table}
        WHERE SEGMENT = :segment_value
        """,
        {"segment_value": SEGMENT},
    )

native_profile.columns = [col.lower() for col in native_profile.columns]
native_profile["snapshot_date"] = pd.to_datetime(native_profile["snapshot_date"])
native_numeric_cols = [
    "memzuc_total_cash_risk_0_24m",
    "fs_net_sales_cumulative",
    "fs_ebitda_cumulative",
    "pos_monthly_volume",
    "ifrs9_behavioral_pd",
    "kkb_commercial_score",
]
native_missing = native_profile[native_numeric_cols].isna().mean().sort_values(ascending=False).reset_index()
native_missing.columns = ["feature", "missing_share"]
snapshot_density = native_profile.groupby("snapshot_date").agg(rows=("customer_id", "size"), customers=("customer_id", "nunique")).reset_index()

display(native_missing)
display(snapshot_density.tail(12))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(snapshot_density["snapshot_date"], snapshot_density["rows"], marker="o")
axes[0].set_title("Native rows by snapshot")
axes[0].tick_params(axis="x", rotation=45)
axes[0].grid(alpha=0.3)
axes[1].bar(native_missing["feature"], native_missing["missing_share"], color="#C44E52")
axes[1].set_title("Native missing share")
axes[1].tick_params(axis="x", rotation=45)
axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for axis, column in zip(axes.flatten(), native_numeric_cols):
    axis.hist(native_profile[column].dropna(), bins=30, color="#4C72B0", alpha=0.85)
    axis.set_title(column)
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 2. Materialization

**What it does**
Runs the real materialization code path from the pipeline to (re)build `EWS_TO_FAZ1_INPUT`:

- `materialize_development` does a full refresh over the complete history.
- `materialize_live` refreshes only the snapshot(s) requested by live scoring.

**How to read the output**

- `enabled: True` means materialization ran. `False` means the config switch is off (see `config/materialization_rules.yaml`).
- `native_rows` should be close to section 1's `row_count`. `derived_rows` is typically smaller because the first 18 snapshots per customer are trimmed as warm-up (needed for `lag_12` features).
- `persisted_rows` equals the number of rows rewritten in `EWS_TO_FAZ1_INPUT`. A value of 0 on development means the frame was empty; on live it means today had no snapshot (expected on mid-month days).
- `quality` sub-dict contains four reports (`native_full`, `native_scope`, `derived_full`, `derived_scope`). Status `pass` / `warn` / `fail` tells whether the data-quality gate cleared. A `fail` raises `QualityGateError` and writes a debug JSON under `runtime/logs/quality/`.

In [ ]:
full_materialization = safe_run(
    "materialize_development",
    materializer.materialize_development,
    SEGMENT,
)
display(pd.DataFrame([compact_dict(full_materialization)]))


In [ ]:
single_materialization = safe_run(
    "materialize_live(single_snapshot)",
    materializer.materialize_live,
    SEGMENT,
    snapshot_date=SINGLE_SNAPSHOT,
)

display(pd.DataFrame([compact_dict(single_materialization)]))
if isinstance(single_materialization, dict) and isinstance(single_materialization.get("frame"), pd.DataFrame):
    display(single_materialization["frame"].head())


## 2A. Derived Quality and Feature Coverage

**What it does**
Queries `EWS_TO_FAZ1_INPUT` for the first six base features and reports:

1. `derived_missing` - missing share per selected feature.
2. `derived_snapshot_density` - rows and unique customers per snapshot.
3. `latest_data_time` - last `DATA_TIME` stamp per snapshot (helps verify the last write time).
4. Plots: rows by snapshot + missing share + per-feature histograms.

**How to read the output**

- Missing share is expected to be non-zero for history-dependent features (`__self_zscore_6`, `__trend_slope_6`, `__population_percentile`) because of warm-up; focus on base columns (for example `bank_debt_to_turnover`).
- Missing share above 25 percent on a base column often indicates that its native source (Memzuc, KKB, FS) was incomplete for that snapshot.
- The `latest_data_time` column should be very close to the snapshot date; a large gap means the derived table was not refreshed recently.
- Histograms should be unimodal and not pathologically skewed; multi-modal shapes appearing only on recent snapshots are early drift indicators.

In [ ]:
with OracleConnector(config, secrets) as ora:
    input_table = ora._qualified_table_name("input_features")
    selected_derived = materializer.base_feature_names[:6]
    derived_profile = ora._read_query(
        f"""
        SELECT SNAPSHOT_DATE, CUSTOMER_ID, DATA_TIME,
               {', '.join(name.upper() for name in selected_derived)}
        FROM {input_table}
        WHERE SEGMENT = :segment_value
        """,
        {"segment_value": SEGMENT},
    )

derived_profile.columns = [col.lower() for col in derived_profile.columns]
derived_profile["snapshot_date"] = pd.to_datetime(derived_profile["snapshot_date"])
derived_profile["data_time"] = pd.to_datetime(derived_profile["data_time"])
derived_missing = derived_profile[selected_derived].isna().mean().sort_values(ascending=False).reset_index()
derived_missing.columns = ["feature", "missing_share"]
derived_snapshot_density = derived_profile.groupby("snapshot_date").agg(rows=("customer_id", "size"), customers=("customer_id", "nunique")).reset_index()
latest_data_time = derived_profile.groupby("snapshot_date")["data_time"].max().reset_index().tail(8)

display(derived_missing)
display(derived_snapshot_density.tail(12))
display(latest_data_time)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(derived_snapshot_density["snapshot_date"], derived_snapshot_density["rows"], marker="o", color="#55A868")
axes[0].set_title("Derived rows by snapshot")
axes[0].tick_params(axis="x", rotation=45)
axes[0].grid(alpha=0.3)
axes[1].bar(derived_missing["feature"], derived_missing["missing_share"], color="#8172B2")
axes[1].set_title("Derived missing share")
axes[1].tick_params(axis="x", rotation=45)
axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for axis, column in zip(axes.flatten(), selected_derived):
    axis.hist(derived_profile[column].dropna(), bins=30, color="#55A868", alpha=0.85)
    axis.set_title(column)
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 3. Development / Training / Calibration

**What it does**
Calls `manager.develop(segment=SEGMENT)`:

- Loads development frames (train / test / calibration / oot).
- Fits the ensemble (Autoencoder, Isolation Forest, Mahalanobis).
- Fits the calibrator (raw score -> percentile) if enabled.
- Registers the model as a candidate and writes the monitoring bundle and stability artefact.

**How to read the output**

- `model_version` is the candidate identifier used by subsequent sections (`tune_weights`, `evaluate_outcomes`, `promote`).
- `monitoring_path` points to `runtime/runs/<run_id>/monitoring/monitoring.json` containing input/score summaries.
- `stability_path` points to `stability.json` used in section 3A.
- `calibration_status` values:
  - `fitted` - calibration trained on enough rows; percentile mapping active.
  - `insufficient_rows` - window smaller than `calibration.min_rows`; scoring falls back to raw scores.
  - `disabled` - calibration explicitly turned off in config.
- `windows` shows the boundary of each split; a missing `oot` boundary means the OOT window is empty.

In [ ]:
develop_summary = safe_run("develop", manager.develop, segment=SEGMENT)
display(pd.DataFrame([develop_summary]))


## 3A. Window Split, Distribution and Stability Review

**What it does**
Reloads the development frames, prints window-level stats (rows / customers / date range), computes per-feature Kolmogorov-Smirnov tests against train, and reads the stability artefact.

**How to read the output**

- `window_summary` table should show four windows (`train`, `test`, `calibration`, `oot`). Empty `oot` is a common cause of failures in section 4.
- `window_feature_stats` shows mean, median, std, missing share per feature per window. Large mean shifts from train to oot signal upstream drift.
- KS test table: `ks_stat < 0.1` is safe, `0.1 - 0.2` is a warning, `> 0.2` implies the model is likely miscalibrated on that window.
- `stability_frame` collates ensemble-level numbers:
  - `ks_stat` - same thresholds as above but on the combined score.
  - `mean_ratio` - `reference_mean / train_mean`; ideal range 0.9 - 1.15.
  - `red_share` - share of customers in the KIRMIZI band for that window.
- The two bar charts visualize `ks_stat` and `mean_ratio` per window; a sudden spike in the OOT bar is the earliest honest signal of model decay.

In [ ]:
frames, windows = manager._load_development_frames(SEGMENT)
window_summary = pd.DataFrame(
    [
        {
            "window": name,
            "rows": len(frame),
            "customers": frame[config['pipeline']['id_column']].nunique() if not frame.empty else 0,
            "snapshot_start": pd.to_datetime(frame[config['pipeline']['time_column']]).min() if not frame.empty else pd.NaT,
            "snapshot_end": pd.to_datetime(frame[config['pipeline']['time_column']]).max() if not frame.empty else pd.NaT,
        }
        for name, frame in frames.items()
    ]
)
display(window_summary)

selected_window_features = materializer.base_feature_names[:4]
window_feature_rows = []
ks_rows = []
train_frame = frames.get('train', pd.DataFrame())
for window_name, frame in frames.items():
    if frame.empty:
        continue
    for feature_name in selected_window_features:
        feature_series = frame[feature_name]
        window_feature_rows.append(
            {
                'window': window_name,
                'feature': feature_name,
                'mean': float(feature_series.mean()),
                'median': float(feature_series.median()),
                'std': float(feature_series.std()),
                'missing_share': float(feature_series.isna().mean()),
            }
        )
        if window_name != 'train' and not train_frame.empty:
            train_values = train_frame[feature_name].dropna()
            ref_values = feature_series.dropna()
            if len(train_values) > 1 and len(ref_values) > 1:
                ks_stat, ks_pvalue = ks_2samp(train_values, ref_values)
                ks_rows.append({'feature': feature_name, 'reference_window': window_name, 'ks_stat': float(ks_stat), 'ks_pvalue': float(ks_pvalue)})

window_feature_stats = pd.DataFrame(window_feature_rows)
display(window_feature_stats)
display(pd.DataFrame(ks_rows))

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for axis, feature_name in zip(axes.flatten(), selected_window_features):
    plot_frame = pd.concat(
        [frame[[feature_name]].assign(window=name) for name, frame in frames.items() if not frame.empty],
        ignore_index=True,
    )
    groups = [group[feature_name].dropna().to_numpy() for _, group in plot_frame.groupby('window')]
    labels = list(plot_frame.groupby('window').groups.keys())
    axis.boxplot(groups, labels=labels, vert=True)
    axis.set_title(feature_name)
    axis.tick_params(axis='x', rotation=45)
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

stability_payload = load_json_payload(develop_summary.get('stability_path')) if isinstance(develop_summary, dict) else None
if stability_payload:
    stability_rows = []
    for window_name, payload in stability_payload.items():
        metric = payload.get('metrics', {}).get('ensemble_score', {})
        stability_rows.append(
            {
                'window': window_name,
                'train_mean': metric.get('train', {}).get('mean'),
                'reference_mean': metric.get('reference', {}).get('mean'),
                'mean_ratio': metric.get('mean_ratio'),
                'ks_stat': metric.get('ks_stat'),
                'ks_pvalue': metric.get('ks_pvalue'),
                'red_share': payload.get('ensemble_alert_share', {}).get('KIRMIZI'),
            }
        )
    stability_frame = pd.DataFrame(stability_rows)
    display(stability_frame)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(stability_frame['window'], stability_frame['ks_stat'], color='#C44E52')
    axes[0].set_title('Ensemble KS by window')
    axes[0].grid(axis='y', alpha=0.3)
    axes[1].bar(stability_frame['window'], stability_frame['mean_ratio'], color='#4C72B0')
    axes[1].set_title('Ensemble mean ratio by window')
    axes[1].grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Stability payload bulunamadi.')


## 3B. Robustness Tests

**What it does**
Runs three side-by-side variants of the pipeline (preprocessing on/off, feature-selection on/off, sampling on/off) using the pipeline's own compare methods. Each comparison lands a JSON + Markdown report under `runtime/runs/<run_id>/`.

**How to read the output**

- `compare_summaries` lists the output paths. If any row has `error`, the comparison failed and the message explains why.
- `compare_metrics` flattens the supervised primary metric for each variant:
  - `baseline_*` - the simpler configuration.
  - `candidate_*` (or `routed_*`, `sampled_*`) - the richer configuration.
  - Positive deltas on precision / recall / f1 mean the extra machinery helps; near-zero deltas mean it does not add much.
- Use these numbers to decide whether preprocessing, feature selection, or sampling should be kept enabled in the live config. A small positive precision delta paired with a small OOT KS drop is usually worth keeping.

In [ ]:
preprocessing_compare = safe_run('compare_preprocessing', manager.compare_preprocessing, segment=SEGMENT)
feature_selection_compare = safe_run('compare_feature_selection', manager.compare_feature_selection, segment=SEGMENT)
sampling_compare = safe_run('compare_sampling', manager.compare_sampling, segment=SEGMENT)
compare_summaries = pd.DataFrame([
    compact_dict(preprocessing_compare),
    compact_dict(feature_selection_compare),
    compact_dict(sampling_compare),
])
display(compare_summaries)

compare_frames = []
for name, summary in [('preprocessing', preprocessing_compare), ('feature_selection', feature_selection_compare), ('sampling', sampling_compare)]:
    payload = load_json_payload(summary.get('comparison_path')) if isinstance(summary, dict) else None
    if not payload:
        continue
    baseline_primary = payload.get('baseline', {}).get('outcomes', {}).get('primary', {})
    candidate_key = [key for key in ('candidate', 'routed', 'sampled') if key in payload]
    candidate_primary = payload.get(candidate_key[0], {}).get('outcomes', {}).get('primary', {}) if candidate_key else {}
    compare_frames.append(
        {
            'comparison': name,
            'baseline_precision': baseline_primary.get('precision_at_top_percent'),
            'baseline_recall': baseline_primary.get('recall_at_top_percent'),
            'baseline_f1': baseline_primary.get('f1_at_top_percent'),
            'candidate_precision': candidate_primary.get('precision_at_top_percent'),
            'candidate_recall': candidate_primary.get('recall_at_top_percent'),
            'candidate_f1': candidate_primary.get('f1_at_top_percent'),
        }
    )
compare_metrics = pd.DataFrame(compare_frames)
display(compare_metrics)


## 4. Supervised Validation

**What it does**
Runs `evaluate_outcomes` (joins scored frames with `EWS_TO_FAZ1_OUTCOMES` and computes labelled metrics) and `tune_weights` (grid-searches ensemble weights on the tuning window with `apply=False`, so production weights are not overwritten).

**How to read the output**

- `evaluation_summary` includes:
  - `evaluation_path` - full JSON artefact.
  - `metrics` - dict with `primary` and `monitoring` sections. `primary` uses `label_30dpd_8w` (configurable).
- `weight_tuning_summary` includes:
  - `weight_version` and `applied` (True if the new weights replaced the defaults).
  - `validation_metrics` at the top percent threshold.
- Both payloads surface:
  - `precision_at_top_percent` - share of the top X percent flagged as positive.
  - `recall_at_top_percent` - share of actual positives captured in the top X percent.
  - `lift_at_top_percent` - precision divided by baseline rate; values above 1.5 are a good sign, close to 1.0 means the ranking is no better than random.

In [ ]:
evaluation_summary = safe_run(
    "evaluate_outcomes",
    manager.evaluate_outcomes,
    segment=SEGMENT,
    model_version=develop_summary.get("model_version") if isinstance(develop_summary, dict) else None,
)
display(pd.DataFrame([evaluation_summary]))

weight_tuning_summary = safe_run(
    "tune_weights",
    manager.tune_weights,
    segment=SEGMENT,
    model_version=develop_summary.get("model_version") if isinstance(develop_summary, dict) else None,
    apply=False,
)
display(pd.DataFrame([weight_tuning_summary]))


## 4A. Supervised Metrics Breakdown

**What it does**
Flattens `metrics` from both the evaluation and weight-tuning payloads into clean DataFrames and draws precision / recall / f1 bar charts for each.

**How to read the output**

- `evaluation_metrics_frame` and `weight_tuning_metrics_frame` break down metrics by `metric_group` (`primary`, per-monitoring-column).
- In the primary row compare:
  - `precision_at_top_percent` against the baseline positive rate - the gap is the real-world value of the model.
  - `recall_at_top_percent` - low is expected because we only mark the top N percent; the absolute number matters less than its direction over time.
  - `f1_at_top_percent` - combined view.
- Bar charts let you eyeball whether the tuned weights clearly beat the default ensemble weights; a difference below 2-3 percent is usually noise.

In [ ]:
evaluation_metrics_frame = flatten_outcome_metrics(evaluation_summary.get('metrics', {})) if isinstance(evaluation_summary, dict) else pd.DataFrame()
weight_tuning_metrics_frame = flatten_outcome_metrics(weight_tuning_summary.get('validation_metrics', {})) if isinstance(weight_tuning_summary, dict) else pd.DataFrame()

display(evaluation_metrics_frame)
display(weight_tuning_metrics_frame)

if not evaluation_metrics_frame.empty:
    draw_metric_bar(
        evaluation_metrics_frame,
        category_column='metric_group',
        value_columns=['precision_at_top_percent', 'recall_at_top_percent', 'f1_at_top_percent'],
        title='Evaluation precision / recall / F1',
    )

if not weight_tuning_metrics_frame.empty:
    draw_metric_bar(
        weight_tuning_metrics_frame,
        category_column='metric_group',
        value_columns=['precision_at_top_percent', 'recall_at_top_percent', 'f1_at_top_percent'],
        title='Weight tuning validation precision / recall / F1',
    )


## 5. Promote and Live Scoring

**What it does**
Promotes the most recent candidate to champion and executes live scoring twice:

- Single snapshot (from `live_scoring.snapshot.selector` or an explicit date).
- Date range (four most recent snapshots, overridable via config).

Both calls go through `engine.lifecycle.LifecycleManager.score_live`, which rematerializes only the requested scope before scoring.

**How to read the output**

- `promote_summary` contains the segment, the promoted model version, and the log path.
- `single_score_summary` / `range_score_summary` include:
  - `status` - `completed`, or `skipped` if no rows existed for that scope (mid-month today() is the common skip cause).
  - `rows` - number of customers scored.
  - `monitoring_path` - per-run JSON with score and band stats.
  - `output` - summary of rows written to `EWS_TO_FAZ1_RESULTS`, `EWS_TO_FAZ1_DETAILS`, `EWS_TO_FAZ1_FEATURE_EFFECTS`.

In [ ]:
promote_summary = safe_run(
    "promote",
    manager.promote,
    segment=SEGMENT,
    model_version=develop_summary.get("model_version") if isinstance(develop_summary, dict) else None,
)
display(pd.DataFrame([promote_summary]))

single_score_summary = safe_run(
    "score_live(single_snapshot)",
    manager.score_live,
    segment=SEGMENT,
    snapshot_date=SINGLE_SNAPSHOT,
)
display(pd.DataFrame([single_score_summary]))

range_score_summary = safe_run(
    "score_live(range)",
    manager.score_live,
    segment=SEGMENT,
    start_date=RANGE_START,
    end_date=RANGE_END,
)
display(pd.DataFrame([range_score_summary]))


## 5A. Scored Range Validation and Label Lift

**What it does**
Joins the range-scoring output with the outcomes table, re-computes the manual top-percent metrics, and summarises the positive rate per alert band.

**How to read the output**

- `manual_top_metrics`:
  - `precision_at_top_percent` - the same metric computed directly against the Oracle results table for sanity.
  - `true_positive` / `false_positive` / `true_negative` / `false_negative` - raw cells of the top-N confusion matrix.
- `band_target` - per band, the count of customers and the mean of the target column. Expected monotonic ordering: `NORMAL < SARI < TURUNCU < KIRMIZI` positive rate. If KIRMIZI does not dominate, the calibration or weights may need attention.
- Left histogram - `anomaly_score` distributions split by label (`0`/`1`). Ideally the label=1 distribution sits clearly to the right of label=0.
- Right bar chart - positive rate per alert band; bars should rise from NORMAL to KIRMIZI.

In [ ]:
with OracleConnector(config, secrets) as ora:
    results_table = ora._qualified_table_name('results')
    outcomes_table = ora._qualified_table_name('outcomes')
    scored_range = ora._read_query(
        f"""
        SELECT r.CUSTOMER_ID, r.SNAPSHOT_DATE, r.ANOMALY_SCORE, r.ALERT_BAND,
               o.{TARGET_COLUMN.upper()} AS target_label
        FROM {results_table} r
        LEFT JOIN {outcomes_table} o
          ON r.CUSTOMER_ID = o.CUSTOMER_ID
         AND TRUNC(r.SNAPSHOT_DATE) = TRUNC(o.SNAPSHOT_DATE)
        WHERE r.SEGMENT = :segment_value
          AND TRUNC(r.SNAPSHOT_DATE) BETWEEN DATE '{RANGE_START}' AND DATE '{RANGE_END}'
        """,
        {'segment_value': SEGMENT},
    )

scored_range.columns = [col.lower() for col in scored_range.columns]
scored_range['snapshot_date'] = pd.to_datetime(scored_range['snapshot_date'])
scored_range['target_label'] = scored_range['target_label'].fillna(0).astype(int)
manual_top_metrics = compute_top_percent_metrics(scored_range, 'anomaly_score', 'target_label', TOP_PERCENT)
display(pd.DataFrame([manual_top_metrics]))

band_target = scored_range.groupby('alert_band')['target_label'].agg(['count', 'mean']).reset_index().rename(columns={'mean': 'positive_rate'})
display(band_target)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for label_value, color in [(0, '#4C72B0'), (1, '#C44E52')]:
    subset = scored_range.loc[scored_range['target_label'] == label_value, 'anomaly_score']
    axes[0].hist(subset, bins=30, alpha=0.6, label=f'label={label_value}', color=color)
axes[0].set_title('Score distribution by target label')
axes[0].legend()
axes[0].grid(alpha=0.2)
axes[1].bar(band_target['alert_band'], band_target['positive_rate'], color='#55A868')
axes[1].set_title('Positive rate by alert band')
axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Monitoring and Oracle Output Review

**What it does**
Reads back the scored snapshot from Oracle:

- Top 5 highest-score customers and their top reasons.
- 10 detail rows (top features explaining an alert).
- A `DATA_TIME` check (when the derived table was last rewritten for this snapshot).

**How to read the output**

- `result_sample` - shows customers with the highest `ANOMALY_SCORE` and the `REASON_1..3` columns. A single feature repeatedly owning all three reasons means the model is leaning heavily on one signal; that is something to investigate.
- `details_sample` - per-feature detail for a scored customer: `ACTUAL_VALUE` vs references, `CONTRIBUTION_PCT`, and component-level contributions (`AE_CONTRIBUTION_PCT`, `IF_CONTRIBUTION_PCT`, `MD_CONTRIBUTION_PCT`). Useful when defending an alert to business users.
- `data_time_check` - `MIN_DATA_TIME` and `MAX_DATA_TIME` should fall inside the current run window; an older timestamp means materialization did not run recently.

In [ ]:
with OracleConnector(config, secrets) as ora:
    results_table = ora._qualified_table_name("results")
    details_table = ora._qualified_table_name("details")
    effects_table = ora._qualified_table_name("full_effects")
    input_table = ora._qualified_table_name("input_features")

    result_sample = ora._read_query(
        f"""
        SELECT CUSTOMER_ID, SNAPSHOT_DATE, ANOMALY_SCORE, ALERT_BAND, REASON_1, REASON_2, REASON_3
        FROM {results_table}
        WHERE TRUNC(SNAPSHOT_DATE) = DATE '{SINGLE_SNAPSHOT}'
          AND ROWNUM <= 5
        ORDER BY ANOMALY_SCORE DESC
        """
    )
    details_sample = ora._read_query(
        f"""
        SELECT CUSTOMER_ID, SNAPSHOT_DATE, FEATURE_NAME, FEATURE_LABEL,
               ACTUAL_VALUE, CUSTOMER_HISTORY_REFERENCE, POPULATION_REFERENCE,
               AE_REFERENCE, CONTRIBUTION_PCT, AE_CONTRIBUTION_PCT, IF_CONTRIBUTION_PCT, MD_CONTRIBUTION_PCT
        FROM {details_table}
        WHERE TRUNC(SNAPSHOT_DATE) = DATE '{SINGLE_SNAPSHOT}'
          AND ROWNUM <= 10
        ORDER BY FEATURE_RANK ASC
        """
    )
    data_time_check = ora._read_query(
        f"""
        SELECT TRUNC(SNAPSHOT_DATE) AS snapshot_date,
               COUNT(*) AS row_count,
               MIN(DATA_TIME) AS min_data_time,
               MAX(DATA_TIME) AS max_data_time
        FROM {input_table}
        WHERE TRUNC(SNAPSHOT_DATE) = DATE '{SINGLE_SNAPSHOT}'
        GROUP BY TRUNC(SNAPSHOT_DATE)
        """
    )

display(result_sample)
display(details_sample)
display(data_time_check)


In [ ]:
monitoring_path = single_score_summary.get("monitoring_path") if isinstance(single_score_summary, dict) else None
if monitoring_path:
    monitoring_payload = json.loads(Path(monitoring_path).read_text(encoding="utf-8"))
    display(pd.DataFrame([monitoring_payload.get("scores", {})]))
    display(pd.DataFrame([monitoring_payload.get("input", {})]))
else:
    print("Monitoring payload bulunamadi.")


## 6A. Monitoring Visualization

**What it does**
Reads the per-run `monitoring.json` produced by the previous score-live call and plots two quick summaries:

- Alert band share bar chart.
- Top missing share of features seen in the scored input.

**How to read the output**

- Band share bars - KIRMIZI between 2 - 10 percent is typical; values above 20 percent indicate the model is over-alerting or data quality dropped.
- Top missing bars - for a healthy live run, these should mostly be the history-dependent columns (`__self_zscore_6`, `__trend_slope_6`). If a base column (like `bank_debt_to_turnover`) appears here the upstream ETL was incomplete.

In [ ]:
if monitoring_path:
    score_payload = monitoring_payload.get('scores', {})
    input_payload = monitoring_payload.get('input', {})
    band_frame = pd.DataFrame(
        [{'alert_band': key, 'share': value} for key, value in (score_payload.get('band_share') or {}).items()]
    )
    missing_frame = pd.DataFrame(
        [{'feature': key, 'missing_share': value} for key, value in (input_payload.get('top_missing_share') or {}).items()]
    )
    display(band_frame)
    display(missing_frame)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    if not band_frame.empty:
        axes[0].bar(band_frame['alert_band'], band_frame['share'], color='#4C72B0')
        axes[0].set_title('Monitoring band share')
        axes[0].grid(axis='y', alpha=0.3)
    if not missing_frame.empty:
        axes[1].bar(missing_frame['feature'], missing_frame['missing_share'], color='#C44E52')
        axes[1].set_title('Top missing share in scored input')
        axes[1].tick_params(axis='x', rotation=45)
        axes[1].grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Monitoring payload bulunamadi.')


## 7A. Monitor History Trend (Oracle)

**What it does**
Reads the last 50 rows from `EWS_TO_FAZ1_MONITOR_HISTORY` (persistent run-level metrics, one row per develop / tune / evaluate / score-live), shows:

1. A raw table of the most recent runs.
2. Four trend plots by run type.
3. A "latest value per run type" summary table.

**How to read the output**

- Recent-runs table - quickly spot any `status` that is not `completed`. `RUN_ID` encodes the run type + segment + timestamp.
- Run type counts - sanity check that the expected run types actually populate the table (every batch should add at least one `score-live`).
- Trend plots (top row uses `score-live` only):
  - **Band share**: KIRMIZI + TURUNCU shares over time. A sudden jump is either an improved signal or a data break - cross-check with section 1A.
  - **Anomaly score stats**: `mean` and `p99`. `p99` > 99 with a tight mean means the distribution is collapsing toward the top - a calibration symptom.
- Trend plots (bottom row):
  - **Develop stability**: ensemble OOT `ks_stat` (safe < 0.1) and `mean_ratio` (ideal 0.9 - 1.15). Drift shows up here first.
  - **Supervised**: `precision@top%` and `lift@top%` across tune-weights and evaluate-outcomes runs. Watch for lift trending toward 1.0.
- Latest-per-type table - single snapshot of current health:
  - `score-live` row shows `score_p99`, `band_kirmizi`, `band_persistence_kirmizi`, `score_psi_vs_prev`, `dominant_reason_feature`.
  - `develop` row shows stability KS per window, calibration rows, monotonic flag.
  - `tune-weights` / `evaluate-outcomes` rows show the latest supervised precision / lift and the active weight mix.

**Health thresholds in one glance**

| metric | green | yellow | red |
|---|---|---|---|
| `band_kirmizi` | 0.02 - 0.10 | 0.10 - 0.20 | > 0.20 or 0 |
| `band_persistence_kirmizi` | > 0.60 | 0.40 - 0.60 | < 0.40 |
| `score_psi_vs_prev` | < 0.1 | 0.1 - 0.25 | > 0.25 |
| `stability_oot_ks` | < 0.10 | 0.10 - 0.20 | > 0.20 |
| `stability_oot_mean_ratio` | 0.90 - 1.15 | 0.85 - 1.20 | outside |
| `supervised_precision` | >= baseline * 1.5 | baseline * 1.1 - 1.5 | <= baseline * 1.1 |
| `supervised_lift` | >= 1.5 | 1.1 - 1.5 | <= 1.1 |
| `dominant_reason_share` | < 0.35 | 0.35 - 0.55 | > 0.55 (single feature dominates) |
| `freshness_max_age_days` | <= 120 | 120 - 220 | > 220 |
| `calibration_monotonic` | 1 | - | 0 or NULL while `calibration_rows` > 0 |

In [ ]:
with OracleConnector(config, secrets) as ora:
    hist_table = ora._qualified_table_name('monitor_history')
    monitor_history = ora._read_query(
        f"""
        SELECT *
        FROM {hist_table}
        WHERE SEGMENT = :segment_value
        ORDER BY FINISHED_AT DESC
        FETCH FIRST 50 ROWS ONLY
        """,
        {'segment_value': SEGMENT},
    )

monitor_history.columns = [col.lower() for col in monitor_history.columns]
monitor_history['finished_at'] = pd.to_datetime(monitor_history['finished_at'])
monitor_history = monitor_history.sort_values('finished_at', ascending=False).reset_index(drop=True)
display(monitor_history.head(20))
display(monitor_history['run_type'].value_counts().rename_axis('run_type').reset_index(name='run_count'))

In [ ]:
if monitor_history.empty:
    print('Monitor history bos.')
else:
    trend = monitor_history.sort_values('finished_at')
    score_trend = trend[trend['run_type'] == 'score-live']
    develop_trend = trend[trend['run_type'] == 'develop']
    supervised_trend = trend[trend['run_type'].isin(['evaluate-outcomes', 'tune-weights'])]

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    if not score_trend.empty:
        axes[0, 0].plot(score_trend['finished_at'], score_trend['band_kirmizi'], marker='o', color='#C44E52', label='KIRMIZI')
        axes[0, 0].plot(score_trend['finished_at'], score_trend['band_turuncu'], marker='s', color='#DD8452', label='TURUNCU')
        axes[0, 0].set_title('Score-live: band share trend')
        axes[0, 0].legend()
        axes[0, 0].tick_params(axis='x', rotation=30)
        axes[0, 0].grid(alpha=0.3)
    else:
        axes[0, 0].set_title('Score-live: (no rows)')

    if not score_trend.empty:
        axes[0, 1].plot(score_trend['finished_at'], score_trend['score_p99'], marker='o', color='#4C72B0', label='p99')
        axes[0, 1].plot(score_trend['finished_at'], score_trend['score_mean'], marker='s', color='#55A868', label='mean')
        axes[0, 1].set_title('Score-live: anomaly_score stats')
        axes[0, 1].legend()
        axes[0, 1].tick_params(axis='x', rotation=30)
        axes[0, 1].grid(alpha=0.3)
    else:
        axes[0, 1].set_title('Score-live: (no rows)')

    if not develop_trend.empty:
        axes[1, 0].plot(develop_trend['finished_at'], develop_trend['stability_oot_ks'], marker='o', color='#8172B2', label='OOT KS')
        axes[1, 0].plot(develop_trend['finished_at'], develop_trend['stability_oot_mean_ratio'], marker='s', color='#937860', label='OOT mean_ratio')
        axes[1, 0].set_title('Develop: stability metrics')
        axes[1, 0].legend()
        axes[1, 0].tick_params(axis='x', rotation=30)
        axes[1, 0].grid(alpha=0.3)
    else:
        axes[1, 0].set_title('Develop: (no rows)')

    if not supervised_trend.empty:
        axes[1, 1].plot(supervised_trend['finished_at'], supervised_trend['supervised_precision'], marker='o', color='#55A868', label='precision@top%')
        axes[1, 1].plot(supervised_trend['finished_at'], supervised_trend['supervised_lift'], marker='s', color='#C44E52', label='lift@top%')
        axes[1, 1].set_title('Supervised: precision & lift')
        axes[1, 1].legend()
        axes[1, 1].tick_params(axis='x', rotation=30)
        axes[1, 1].grid(alpha=0.3)
    else:
        axes[1, 1].set_title('Supervised: (no rows)')

    plt.tight_layout()
    plt.show()

In [ ]:
if monitor_history.empty:
    print('Monitor history bos, run tipi bazinda ozet uretilmedi.')
else:
    latest_per_type = (
        monitor_history.sort_values('finished_at', ascending=False)
        .groupby('run_type', as_index=False)
        .first()
    )
    display_columns = [
        'run_type', 'finished_at', 'model_version', 'status',
        'input_rows', 'band_kirmizi', 'score_p99',
        'quality_native_full', 'quality_derived_scope',
        'stability_oot_ks', 'stability_oot_mean_ratio',
        'supervised_precision', 'supervised_lift',
        'weight_ae', 'weight_if', 'weight_md',
    ]
    present = [col for col in display_columns if col in latest_per_type.columns]
    summary_frame = latest_per_type[present].copy()
    for col in summary_frame.columns:
        if summary_frame[col].dtype == object:
            summary_frame[col] = summary_frame[col].fillna('-')
    display(summary_frame)

## 7. Notes

- Native tablo ayni contract ile geldiginde notebook tekrar kullanilabilir.
- Derived/input rematerialization score cagrisi icinde otomatik yapilir.
- Single-date score ayni snapshot icin tekrar kosarsa input tablosundaki `DATA_TIME` guncellenir.
- Range scoring output tablolarinda ilgili tarih araligini replace eder.
- Scheduler bu notebook icinden degil, dis orchestration araci uzerinden tetiklenmelidir.
- Sorun giderme: bir hucre hata verirse `runtime/logs/` altindaki ilgili run log'una ve (quality fail ise) `runtime/logs/quality/` altindaki dump JSON dosyasina bakin. Tum run meta + score trend bilgisi Oracle `EWS_TO_FAZ1_MONITOR_HISTORY` tablosundan da okunabilir.